# Финальное решение: детекция ботов по поведению cookie

## Цель

Для каждой `cookie_id` необходимо построить непрерывный `score` в диапазоне `[0, 1]`: чем выше значение, тем выше вероятность того, что трафик относится к автоматизированному сборщику данных.

Финальный файл должен содержать ровно две колонки - `cookie_id` и `score` - и одну строку для каждой cookie из `test.csv`.

---

## Подход к решению

Я рассматриваю задачу как **бинарную классификацию на уровне cookie**, где одна строка train/test соответствует одной 24-часовой наблюдаемой сессии cookie.

Основная идея состоит в том, чтобы не использовать отдельные события как независимые объекты классификации, а сначала агрегировать всю историю событий внутри разрешенного окна наблюдения в набор поведенческих признаков.

Пайплайн состоит из следующих этапов:

1. **Проверка входных данных.** Проверяю схему таблиц, уникальность `cookie_id`, отсутствие пересечения train/test и корректность 24-часовых окон.
2. **Фильтрация событий по окну.** Для каждой cookie оставляю только события, удовлетворяющие `window_start_ts <= event_ts < window_end_ts`. Это важно для исключения утечки информации из будущего.
3. **Feature engineering.** Строю признаки четырех типов:
   - интенсивность и разнообразие активности;
   - количество событий каждого типа;
   - доли событий каждого типа;
   - временные характеристики поведения и регулярности событий.
4. **Контроль схемы признаков.** Train и test приводятся к одной детерминированной числовой схеме. Cookie без событий не удаляются: после `left join` их агрегаты заполняются нулями.
5. **Модель.** Используется `CatBoostClassifier`, выбранный по walk-forward validation на исторических датах.
6. **Финальное обучение.** После выбора признаков и модели классификатор обучается на всем размеченном train.
7. **Submission.** В test сохраняются вероятности класса 1 через `predict_proba(... )[:, 1]`. Порог классификации не применяется, поскольку проверяющий сам выбирает порог при расчете `Precision @ Recall >= 0.70`.

### Почему именно такой validation

Обычный случайный train/validation split здесь нежелателен: test относится к более позднему периоду времени. Поэтому итоговый выбор модели выполнялся через **walk-forward validation**: для каждой validation date модель обучалась только на более ранних датах.

По pooled `P@R >= 0.70` на внутренних validation-данных получили:

| Модель | P@R >= 0.70 | PR-AUC | ROC-AUC |
|---|---:|---:|---:|
| CatBoost | **0.4575** | 0.6695 | 0.8856 |
| HistGradientBoosting | 0.4427 | 0.6807 | 0.8861 |
| Random Forest, `min_samples_leaf=10` | 0.4308 | 0.5955 | 0.8721 |

Основным критерием выбора был именно `P@R >= 0.70`, потому что это официальная метрика задания. Значение `0.4575` - **только результат внутренней validation**, а не результат скрытого test.

## Подход к решению

Задача рассматривается как **бинарная классификация на уровне `cookie_id`**. Для каждой cookie нужно оценить вероятность того, что ее трафик относится к автоматизированному сборщику данных. Поэтому на вход модели должна поступать одна фиксированная строка признаков на каждую cookie, а на выходе - непрерывный `score` в диапазоне `[0, 1]`.

### 1. Ограничиваем данные индивидуальным окном наблюдения

Для каждой строки `train` и `test` задано собственное 24-часовое окно. Сначала присоединяем его границы к событиям по `cookie_id`, а затем оставляем только события, удовлетворяющие:

`window_start_ts <= event_ts < window_end_ts`

Это принципиальный шаг: события за пределами окна не должны использоваться при построении признаков, иначе в признаки может попасть информация, недоступная в момент принятия решения.

### 2. Переводим историю событий в признаки уровня cookie

Одна cookie может иметь произвольное количество событий, поэтому напрямую передать event-level таблицу в обычный классификатор нельзя. История сжимается в фиксированный набор числовых признаков.

Используются несколько независимых групп:

- **метаданные** - возраст cookie на начало окна;
- **интенсивность и разнообразие** - общее число событий, количество уникальных объектов, категорий, локаций, поисковых запросов и типов событий;
- **счетчики событий** - сколько раз каждый тип действия встретился;
- **доли событий** - какая часть всей активности приходится на каждый тип действия;
- **временные признаки** - начало и конец активности, число активных часов, статистики интервалов между событиями и характеристика burstiness.

Так модель получает не отдельные события, а компактное числовое описание поведения cookie за наблюдаемый период.

### 3. Сохраняем одинаковую схему train и test

Набор `EVENT_TYPES` фиксирован заранее. Это гарантирует, что при `crosstab` отсутствие какого-либо типа события в конкретной выборке не изменит структуру матрицы.

После объединения признаков:

- удаляется дублирующий `cookie_age_seconds`;
- объединяется возможное дублирование `n_events`;
- остаются только числовые признаки;
- test принудительно приводится к схеме признаков train.

Cookie без событий также не теряются: метаданные являются основной таблицей, а event-derived признаки после `left join` заполняются нулями.

### 4. Проверяем корректность данных до обучения

В pipeline есть проверки:

- обязательных колонок;
- уникальности `cookie_id`;
- отсутствия пересечения train и test;
- корректности 24-часовых окон;
- отсутствия событий за пределами окна;
- сохранения полного набора cookie после feature engineering;
- отсутствия `NaN` и бесконечных значений;
- корректной структуры submission.

Таким образом, проверки являются частью самого решения, а не только ручной диагностикой в notebook.

### 5. Обучаем финальную модель на всем train

В исследовании сравнивались разные модели и варианты признаков с временным разбиением. Финальной выбрана `CatBoostClassifier` с зафиксированными параметрами.

После завершения validation модель обучается на всем размеченном `train`, поскольку test-разметка недоступна. Для `test` вычисляется `predict_proba(... )[:, 1]`.

Порог классификации намеренно не используется. Задание требует непрерывный `score`, а официальная метрика оценивает ранжирование объектов и сама рассматривает пороги при достижении требуемого recall.

Итоговая цепочка решения:

**события → фильтрация по 24-часовому окну → cookie-level признаки → числовая feature matrix → CatBoost → непрерывный score → `submission.csv`**.


## Важные ограничения, которые соблюдаются в решении

- не используются внешние API;
- не используются LLM или внешние модели;
- не используются ручные метки для test;
- не используются конкретные `cookie_id` как ручные правила;
- признаки строятся только из данных, доступных внутри 24-часового окна;
- threshold не фиксируется и не записывается в submission;
- итоговая модель обучается на всем размеченном train после завершения validation.

`metric.py` из задания используется в исследовательском ноутбуке для расчета официальной метрики.

In [39]:
# Включаем поддержку отложенной обработки аннотаций типов, чтобы типы функций не требовали немедленного разрешения.
from __future__ import annotations

# Path используется для построения путей независимо от операционной системы и делает расположение данных относительно проекта явным.
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

# Фиксируем seed, чтобы исследование и финальное обучение использовали одинаковую случайную инициализацию.
SEED = 42
# Если ноутбук запущен из notebooks/ — поднимаемся на уровень выше.
CWD = Path.cwd()
if CWD.name == "notebooks":
    ROOT_DIR = CWD.parent
else:
    ROOT_DIR = CWD  # fallback, если запускают из корня проекта

DATA_DIR = ROOT_DIR / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
EVENTS_PATH = DATA_DIR / "events.csv.gz"
SUBMISSION_PATH = ROOT_DIR / "submission.csv"

# Фиксируем полный набор типов событий заранее. Это делает структуру признаков детерминированной и одинаковой для train и test.
EVENT_TYPES = [
    # Каждый элемент списка соответствует отдельному типу поведения, для которого будут построены счетчик и доля.
    "item_view",
    "search_results_view",
    "photo_swipe",
    "favorite_add",
    "seller_page_view",
    "contact_phone_show",
    "login",
    "contact_chat_open",
    "contact_message_sent",
]

# Задаем обязательные поля train, чтобы дальнейшие операции работали с ожидаемой схемой.
REQUIRED_TRAIN_COLUMNS = {
    "cookie_id",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
    "target",
}

# Для test target не требуется, поэтому набор обязательных колонок отличается от train.
REQUIRED_TEST_COLUMNS = {
    "cookie_id",
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
}

# Эта схема описывает минимальный набор полей, необходимый для фильтрации и feature engineering по событиям.
REQUIRED_EVENT_COLUMNS = {
    "cookie_id",
    "event_ts",
    "eid",
    "event_name",
    "platform",
    "user_agent",
    "item_id",
    "item_category",
    "item_location",
    "seller_type",
    "search_query",
    "search_page",
    "pointer_x",
    "pointer_y",
}

In [40]:
# Выделяем загрузку данных в отдельную функцию: чтение файлов не смешивается с подготовкой признаков.
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three input tables with the required datetime columns parsed."""

    # Сначала проверяем наличие всех трех файлов, чтобы получить понятную ошибку еще до чтения данных.
    for path in (TRAIN_PATH, TEST_PATH, EVENTS_PATH):
        # Если файл отсутствует, выполнение останавливается сразу с указанием конкретного пути.
        if not path.exists():
            # Явно сообщаем, какой обязательный входной файл не найден.
            raise FileNotFoundError(
                f"Required input file was not found: {path}"
            )

    # Читаем train и сразу преобразуем временные колонки в datetime, чтобы далее выполнять корректные операции со временем.
    train = pd.read_csv(
        TRAIN_PATH,
        parse_dates=[
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ],
    )

    # Аналогично загружаем test с теми же временными полями, но без target.
    test = pd.read_csv(
        TEST_PATH,
        parse_dates=[
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ],
    )

    # Загружаем event-level таблицу; для нее достаточно распарсить event_ts.
    events = pd.read_csv(
        EVENTS_PATH,
        parse_dates=["event_ts"],
    )

    # Возвращаем три исходные таблицы без изменения их содержимого.
    return train, test, events

## 1. Фиксируем схему данных

Явное описание обязательных колонок позволяет сразу обнаружить поврежденный или неполный входной файл, а не получить ошибку намного позже во время feature engineering.

Список `EVENT_TYPES` также фиксирован. Это нужно для того, чтобы train и test гарантированно получили одинаковый набор колонок после `crosstab`, даже если в конкретной подвыборке какой-либо тип события отсутствует.

In [41]:
# Проверяем не только наличие колонок, но и ключевые инварианты задачи до начала feature engineering.
def validate_input(
        train: pd.DataFrame,
        test: pd.DataFrame,
        events: pd.DataFrame,
) -> None:
    """Validate the input schema and invariants required by the task."""

    # Вычисляем, каких обязательных колонок не хватает в train.
    missing_train = REQUIRED_TRAIN_COLUMNS - set(train.columns)

    # Отдельно проверяем схему test.
    missing_test = REQUIRED_TEST_COLUMNS - set(test.columns)

    # Отдельно проверяем схему event-level данных.
    missing_events = REQUIRED_EVENT_COLUMNS - set(events.columns)

    # При нарушении схемы не продолжаем, чтобы не получить скрытую ошибку позже.
    if missing_train:
        raise ValueError(f"Missing train columns: {sorted(missing_train)}")
    if missing_test:
        raise ValueError(f"Missing test columns: {sorted(missing_test)}")
    if missing_events:
        raise ValueError(f"Missing event columns: {sorted(missing_events)}")

    # Одна строка train должна соответствовать одной cookie.
    assert train["cookie_id"].is_unique

    # То же требование проверяем для test.
    assert test["cookie_id"].is_unique

    # Train и test должны содержать разные cookie, иначе возможна утечка между выборками.
    assert set(train["cookie_id"]).isdisjoint(set(test["cookie_id"]))

    # Target должен быть бинарным и содержать только классы 0 и 1.
    assert train["target"].isin([0, 1]).all()

    # Проверяем, что конец каждого окна действительно позже начала.
    # Проверяем точную продолжительность окна: по условию она равна 24 часам.
    assert (train["window_end_ts"] > train["window_start_ts"]).all()

    # То же проверяем для test.
    assert (test["window_end_ts"] > test["window_start_ts"]).all()

    assert (
            (train["window_end_ts"] - train["window_start_ts"])
            == pd.Timedelta(days=1)
    ).all()
    assert (
            (test["window_end_ts"] - test["window_start_ts"])
            == pd.Timedelta(days=1)
    ).all()

In [42]:
# Главная защита от временной утечки: для каждой cookie оставляем только события внутри ее собственного окна наблюдения.
def events_in_window(
        events: pd.DataFrame,
        meta: pd.DataFrame,
) -> pd.DataFrame:
    """Keep only events belonging to each cookie's half-open observation window."""

    # Из метаданных берем только идентификатор cookie и границы окна, необходимые для сопоставления с событиями.
    meta_window = meta[
        ["cookie_id", "window_start_ts", "window_end_ts"]
    ]

    # Присоединяем границы окна к каждому событию по cookie_id.
    ev = events.merge(
        meta_window,

        # Связываем событие именно с окном той же cookie.
        on="cookie_id",

        # Сохраняем только события, для которых существует соответствующая cookie в текущей выборке.
        how="inner",

        # Проверяем структуру связи: много событий могут относиться к одной cookie, но окно у cookie должно быть единственным.
        validate="many_to_one",
    )

    # Строим маску допустимых событий по полуинтервалу [window_start_ts, window_end_ts).
    mask = (
        # Левая граница включается: событие в момент начала окна доступно модели.
            (ev["event_ts"] >= ev["window_start_ts"])

            # Правая граница исключается, поэтому событие ровно в конце окна уже не используется.
            & (ev["event_ts"] < ev["window_end_ts"])
    )

    # После фильтрации создаем независимую таблицу событий, которая дальше используется только для признаков.
    ev = ev.loc[mask].copy()

    # Сортируем события для детерминированного расчета временных разниц.
    ev = (
        # Сначала группируем по cookie, затем по времени; одинаковые timestamps разрешаются через eid.
        ev.sort_values(
            # eid выступает стабильным tie-breaker для событий с одинаковым временем.
            ["cookie_id", "event_ts", "eid"],

            # Стабильная сортировка сохраняет детерминированный порядок при равных ключах.
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # Нормализуем платформу только для агрегированного признака разнообразия платформ.
    ev["platform_normalized"] = (
        # Приводим значения к строковому виду, убираем пробелы и нормализуем регистр, чтобы web/WEB/Web считались одной платформой.
        ev["platform"].astype(str).str.strip().str.lower()
    )

    # Возвращаем только события, которые одновременно удовлетворяют временному ограничению и имеют детерминированный порядок.
    return ev

## 2. Загрузка данных

Даты сразу преобразуются в `datetime`. Это позволяет безопасно сравнивать `event_ts` с границами окна и вычислять возраст cookie и временные интервалы между событиями.

In [43]:
# После фильтрации повторно проверяем границы окна уже на готовых таблицах train и test.
def validate_events_in_window(
        ev_train: pd.DataFrame,
        ev_test: pd.DataFrame,
) -> None:
    """Assert that no event outside its cookie's observation window remains."""

    # Одинаковая проверка выполняется для обеих выборок.
    for ev in (ev_train, ev_test):
        # Любое событие за пределами собственного окна должно немедленно остановить pipeline.
        assert (
                (ev["event_ts"] >= ev["window_start_ts"])
                & (ev["event_ts"] < ev["window_end_ts"])
        ).all()

In [44]:
# Строим признаки из информации, доступной еще до начала наблюдаемого периода.
def add_metadata_features(meta: pd.DataFrame) -> pd.DataFrame:
    """Build features available from cookie metadata before the event window."""

    # Оставляем cookie_id для последующих объединений и временные поля, необходимые для расчета возраста.
    result = meta[
        [
            # Идентификатор используется только как ключ объединения и не передается модели как признак.
            "cookie_id",
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()

    # Считаем возраст cookie на момент начала окна: это информация, доступная без просмотра будущих событий.
    result["cookie_age_seconds"] = (
        # Разница между началом окна и созданием cookie дает ее возраст в секундах.
            result["window_start_ts"] - result["cookie_created_at"]
    ).dt.total_seconds()
    # Переводим тот же показатель в часы; именно эту форму сохраняем в финальной матрице.
    result["cookie_age_hours"] = result["cookie_age_seconds"] / 3600.0

    # Возвращаем таблицу метаданных, которая затем станет базой для всех объединений.
    return result

## 3. Ограничение событий разрешенным окном наблюдения

Ключевое правило задачи - использовать только события внутри индивидуального окна конкретной cookie.

Используется полуинтервал:

`window_start_ts <= event_ts < window_end_ts`

Такой вариант однозначно задает принадлежность события границе соседних окон.

После фильтрации события сортируются по `cookie_id`, времени и `eid`. Стабильная сортировка `mergesort` нужна для воспроизводимого порядка событий при одинаковых timestamp.

In [45]:
# Сводим переменное число событий каждой cookie к фиксированному набору числовых агрегатов.
def build_activity_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Aggregate event volume and behavioural diversity to cookie level."""

    # Группировка делает все последующие расчеты независимыми для каждой cookie.
    g = ev.groupby("cookie_id", sort=False)

    features = pd.DataFrame(
        {
            # Общее количество событий отражает интенсивность активности внутри окна.
            "n_events": g.size(),

            # Количество уникальных объектов показывает разнообразие просмотренного контента.
            "n_unique_items": g["item_id"].nunique(),

            # Разнообразие категорий помогает отличать узкое повторяющееся поведение от широкого просмотра.
            "n_unique_categories": g["item_category"].nunique(),

            # Считаем число уникальных локаций, встречавшихся в событиях cookie.
            "n_unique_locations": g["item_location"].nunique(),

            # Считаем разнообразие поисковых запросов внутри окна.
            "n_unique_search_queries": g["search_query"].nunique(),

            # Количество типов действий характеризует широту поведенческого сценария.
            "n_unique_event_types": g["event_name"].nunique(),

            # Используем нормализованные платформы, чтобы различия в регистре не создавали ложную категорию.
            "n_unique_platforms": g["platform_normalized"].nunique(),
        }
    )

    # Возвращаем cookie_id из индекса в обычную колонку для последующих merge.
    return features.reset_index()

In [46]:
# Здесь сохраняем абсолютную частоту каждого типа события отдельно.
def build_event_count_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Build per-cookie counts for each event type."""

    # Строим матрицу: строки - cookie, столбцы - типы событий, значения - число событий.
    counts = pd.crosstab(ev["cookie_id"], ev["event_name"])

    # Принудительно задаем полный и одинаковый набор столбцов EVENT_TYPES для train и test.
    # Если конкретного типа события нет у cookie или в текущей выборке, его количество должно быть равно нулю.
    counts = counts.reindex(columns=EVENT_TYPES, fill_value=0)

    # Добавляем префикс, чтобы имена счетчиков были однозначны и отличались от других групп признаков.
    counts.columns = [f"event_count__{event_type}" for event_type in EVENT_TYPES]

    # Возвращаем cookie_id из индекса и получаем обычную таблицу признаков.
    return counts.reset_index()

## 4. Признаки из метаданных cookie

Возраст cookie на начало наблюдаемого окна может отражать различия между недавно созданными и давно существующими cookie.

В модель передается возраст в часах. Секундное представление сохранялось во время исследования, но в финальной схеме было удалено как дублирующее по смыслу.

In [47]:
# Здесь переводим абсолютные количества событий в относительные доли.
def build_event_share_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Build per-cookie event-type shares."""

    # Сначала снова получаем количества каждого типа события по cookie.
    counts = pd.crosstab(ev["cookie_id"], ev["event_name"])

    # Используем тот же фиксированный EVENT_TYPES, чтобы схема не зависела от фактически встреченных столбцов.
    counts = counts.reindex(columns=EVENT_TYPES, fill_value=0)

    # Делим каждую строку на ее общее число событий, поэтому сумма долей одной cookie равна единице при наличии событий.
    shares = counts.div(counts.sum(axis=1), axis=0)

    # Если событий нет и деление дает NaN, отсутствие активности представляем нулевыми значениями.
    shares = shares.fillna(0.0)

    # Префикс event_share__ отделяет относительные признаки от абсолютных счетчиков.
    shares.columns = [f"event_share__{event_type}" for event_type in EVENT_TYPES]

    # Возвращаем cookie_id в колонку для последующего объединения.
    return shares.reset_index()

## 5. Признаки интенсивности и разнообразия поведения

Для каждой cookie агрегируется вся история событий внутри окна.

Здесь важно не просто посчитать число действий, но и измерить **разнообразие** поведения:

- сколько разных объявлений просмотрено;
- сколько разных категорий и локаций встречалось;
- сколько разных поисковых запросов использовано;
- сколько типов событий встречается;
- на скольких платформах происходила активность.

Такие признаки переводят event-level историю в компактное cookie-level представление.

In [48]:
# Временные признаки описывают не только количество действий, но и ритм поведения внутри 24-часового окна.
def build_temporal_features(ev: pd.DataFrame) -> pd.DataFrame:
    """Aggregate timing, activity span and inter-event regularity features."""

    # Берем только поля, необходимые для временных расчетов, не изменяя исходную event-level таблицу.
    df = ev[
        [
            "cookie_id",
            "event_ts",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()

    # Переводим абсолютное время события в смещение от начала индивидуального окна cookie.
    df["event_offset_seconds"] = (
        # Такой расчет делает временной признак сопоставимым между окнами, начинающимися в разное календарное время.
            df["event_ts"] - df["window_start_ts"]
    ).dt.total_seconds()

    # Считаем интервалы между соседними событиями каждой cookie после предварительной сортировки.
    df["inter_event_seconds"] = (
        df.groupby("cookie_id", sort=False)["event_ts"]
        # Первое событие каждой cookie не имеет предыдущего события, поэтому для него возникает NaN.
        .diff()
        .dt.total_seconds()
    )

    # Переводим смещение в номер часа внутри наблюдаемого окна.
    df["relative_hour"] = (df["event_offset_seconds"] // 3600).astype(int)

    # Группируем временную историю по cookie для получения итоговых статистик.
    g = df.groupby("cookie_id", sort=False)

    # Одновременно считаем положение активности в окне, ее продолжительность и статистики интервалов между событиями.
    features = g.agg(
        # Первое событие показывает, насколько быстро cookie начинает активность после начала окна.
        first_event_offset_seconds=("event_offset_seconds", "min"),

        # Последнее событие показывает, насколько далеко по окну продолжается активность.
        last_event_offset_seconds=("event_offset_seconds", "max"),

        # Число активных часов показывает, насколько широко по времени распределена активность.
        n_active_hours=("relative_hour", "nunique"),

        # Средний интервал характеризует типичный темп последовательных действий.
        mean_inter_event_seconds=("inter_event_seconds", "mean"),

        # Медиана менее чувствительна к отдельным очень длинным паузам.
        median_inter_event_seconds=("inter_event_seconds", "median"),

        # Стандартное отклонение показывает вариативность интервалов.
        std_inter_event_seconds=("inter_event_seconds", "std"),

        # Минимальный интервал отражает самые быстрые последовательные действия.
        min_inter_event_seconds=("inter_event_seconds", "min"),

        # Максимальный интервал показывает наиболее длинную паузу между событиями.
        max_inter_event_seconds=("inter_event_seconds", "max"),
    ).reset_index()

    # Вычисляем фактическую продолжительность активности между первым и последним событием.
    features["active_span_seconds"] = (
            features["last_event_offset_seconds"]
            - features["first_event_offset_seconds"]
    )

    # Сохраняем число событий и здесь, потому что оно нужно для нормировки активности по длительности.
    n_events = g.size().rename("n_events").reset_index()

    # Присоединяем число событий к временным агрегатам; one-to-one подтверждает одну строку на cookie.
    features = features.merge(
        n_events,
        on="cookie_id",
        how="left",
        validate="one_to_one",
    )

    # Получаем интенсивность событий относительно длительности активного периода.
    features["events_per_active_hour"] = features["n_events"] / (
            features["active_span_seconds"] / 3600.0 + 1.0
    )

    # Формируем знаменатель для нормированной характеристики вариативности интервалов.
    denominator = (
            features["std_inter_event_seconds"]
            + features["mean_inter_event_seconds"]
    )

    # Строим итоговую характеристику burstiness из среднего и стандартного отклонения интервалов.
    features["inter_event_burstiness"] = (
                                                 features["std_inter_event_seconds"]
                                                 - features["mean_inter_event_seconds"]
                                         ) / denominator.replace(0, np.nan)

    # Явно перечисляем интервальные признаки, для которых допустимы NaN из-за отсутствия пары событий.
    interval_cols = [
        "mean_inter_event_seconds",
        "median_inter_event_seconds",
        "std_inter_event_seconds",
        "min_inter_event_seconds",
        "max_inter_event_seconds",
        "inter_event_burstiness",
    ]

    # Заменяем неопределенные значения нулем, чтобы итоговая матрица не содержала пропусков.
    features[interval_cols] = features[interval_cols].fillna(0.0)

    # Возвращаем одну временную строку признаков на каждую cookie, представленную в event-таблице.
    return features

## 6. Распределение типов событий

Одно и то же число событий может соответствовать совершенно разному поведению. Поэтому дополнительно считаю:

1. абсолютное количество каждого типа события;
2. долю каждого типа среди всех событий cookie.

Например, последовательность из просмотров объявлений, поисков и действий с контактами может иметь другую структуру, чем поток однотипных действий.

In [49]:
# Объединяем независимые семейства признаков в одну таблицу уровня cookie.
def build_feature_table(
        meta: pd.DataFrame,
        ev: pd.DataFrame,
) -> pd.DataFrame:
    """Build the complete cookie-level feature table."""

    # Список фиксирует состав финального feature engineering и его порядок.
    parts = [
        # Метаданные задают базовую таблицу и полный набор cookie.
        add_metadata_features(meta),

        # Добавляем общие показатели интенсивности и разнообразия.
        build_activity_features(ev),

        # Добавляем абсолютные количества типов событий.
        build_event_count_features(ev),

        # Добавляем относительную структуру поведения.
        build_event_share_features(ev),

        # Добавляем временной ритм и регулярность активности.
        build_temporal_features(ev),
    ]

    # Начинаем с метаданных, чтобы последующие left join не удаляли cookie без событий.
    features = parts[0].copy()

    # Последовательно присоединяем остальные семейства признаков.
    for part in parts[1:]:
        # Используем left join: исходный набор cookie должен сохраняться полностью.
        features = features.merge(
            part,
            on="cookie_id",
            how="left",

            # Проверяем, что каждый блок действительно содержит не более одной строки на cookie.
            validate="one_to_one",
        )

    # Получаем полную сырую таблицу признаков до финальной канонизации.
    return features

## 7. Временные признаки

Временная структура - одна из ключевых частей решения.

Для каждой cookie вычисляются:

- время первого и последнего события относительно начала окна;
- число активных часов;
- средний, медианный и стандартный inter-event time;
- минимальный и максимальный интервал между соседними событиями;
- продолжительность активного периода;
- число событий на активный час;
- `inter_event_burstiness` - нормированная характеристика регулярности/вариативности интервалов.

Первое событие для cookie не имеет предыдущего события, поэтому его inter-event time равен `NaN`; такие значения заполняются нулем после агрегации.

In [50]:
# Приводим сырую feature table к единой детерминированной числовой схеме, которую получит модель.
def canonicalize_features(
        X_train_raw: pd.DataFrame,
        X_test_raw: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Create one deterministic numeric feature schema for train and test."""

    # Работаем с копиями, чтобы не менять исходные таблицы признаков.
    train_df = X_train_raw.copy()

    # Для test выполняем те же преобразования, что и для train.
    test_df = X_test_raw.copy()

    # Убираем дублирующее представление возраста; в финальных экспериментах оставляли возраст в часах.
    train_df = train_df.drop(columns=["cookie_age_seconds"])

    # То же преобразование обязательно выполняем для test.
    test_df = test_df.drop(columns=["cookie_age_seconds"])

    # Ищем все колонки, связанные с n_events, потому что merge мог создать суффиксированные копии.
    n_events_candidates = [
        col for col in train_df.columns if col.startswith("n_events")
    ]

    # Без признака общей активности невозможно восстановить ожидаемую финальную схему.
    if not n_events_candidates:
        # Останавливаем pipeline с понятной причиной, если обязательный сигнал отсутствует.
        raise ValueError("No n_events-derived feature is available")

    # Если канонического имени еще нет, выбираем один из эквивалентных источников.
    if "n_events" not in train_df.columns:
        # Берем первый доступный вариант; в текущем pipeline дублирующие n_events имеют одинаковое содержание.
        source = n_events_candidates[0]

        # Создаем единое имя признака для train.
        train_df["n_events"] = train_df[source]

        # Создаем такое же имя и для test.
        test_df["n_events"] = test_df[source]

        # Добавляем каноническое имя в список, чтобы старые варианты затем удалить.
        n_events_candidates.append("n_events")

    # Определяем все копии n_events, кроме канонического столбца.
    duplicate_n_events = [
        col for col in n_events_candidates if col != "n_events"
    ]

    # Удаляем лишние копии из train.
    train_df = train_df.drop(columns=duplicate_n_events, errors="ignore")

    # И симметрично удаляем их из test.
    test_df = test_df.drop(columns=duplicate_n_events, errors="ignore")

    # Формируем окончательный список признаков только из числовых колонок.
    feature_cols = [
        col
        for col in train_df.columns
        if col != "cookie_id"
           # Это исключает временные и прочие нечисловые служебные поля из входа CatBoost.
           and pd.api.types.is_numeric_dtype(train_df[col])
    ]

    # Фиксируем порядок train как cookie_id плюс окончательный список признаков.
    train_df = train_df[["cookie_id"] + feature_cols]

    # Принудительно приводим test к точно такой же схеме и порядку колонок.
    test_df = test_df.reindex(columns=["cookie_id"] + feature_cols)

    # Проверяем, что после канонизации train действительно содержит весь ожидаемый набор.
    missing_train_columns = set(feature_cols) - set(train_df.columns)

    # Та же проверка выполняется для test.
    missing_test_columns = set(feature_cols) - set(test_df.columns)

    # Любое расхождение схемы лучше обнаружить до обучения, а не получить неявную ошибку модели.
    if missing_train_columns or missing_test_columns:
        raise ValueError(
            "Feature schema mismatch: "
            f"missing_train={sorted(missing_train_columns)}, "
            f"missing_test={sorted(missing_test_columns)}"
        )

    # Возвращаем подготовленные матрицы и единый список признаков, используемый при fit и predict.
    return train_df, test_df, feature_cols

## 8. Объединение всех признаков

Каждый блок feature engineering возвращает таблицу с одной строкой на cookie. Далее таблицы объединяются через `left join` с метаданными.

Именно `left join` принципиален: если у cookie нет ни одного события внутри окна, такая cookie все равно должна попасть в train/test feature matrix.

In [51]:
# Заполняем пропуски после left join и выполняем финальные проверки матриц перед обучением.
def fill_and_validate_features(
        X_train: pd.DataFrame,
        X_test: pd.DataFrame,
        feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fill aggregation NaNs for cookies without events and validate matrices."""

    # Создаем копию, чтобы функция не изменяла переданный объект снаружи.
    X_train = X_train.copy()

    # То же делаем для тестовой матрицы.
    X_test = X_test.copy()

    # Для cookie без событий event-derived агрегаты отсутствуют; нулевая активность кодируется нулем.
    X_train[feature_cols] = X_train[feature_cols].fillna(0.0)

    # Аналогично обрабатываем test, сохраняя одинаковую семантику признаков.
    X_test[feature_cols] = X_test[feature_cols].fillna(0.0)

    # Проверяем, что train имеет ровно ожидаемую структуру: ключ и feature_cols.
    assert list(X_train.columns) == ["cookie_id"] + feature_cols

    # То же проверяем для test.
    assert list(X_test.columns) == ["cookie_id"] + feature_cols

    # Защищаемся от случайно пустой обучающей выборки.
    assert X_train.shape[0] > 0

    # Тест также должен содержать хотя бы одну cookie.
    assert X_test.shape[0] > 0

    # После заполнения в train не должно остаться пропусков.
    assert X_train[feature_cols].isna().sum().sum() == 0

    # То же проверяем для test.
    assert X_test[feature_cols].isna().sum().sum() == 0

    # Проверяем отсутствие NaN и бесконечностей в числовой матрице train.
    assert np.isfinite(X_train[feature_cols].to_numpy()).all()

    # Аналогично проверяем test.
    assert np.isfinite(X_test[feature_cols].to_numpy()).all()

    # Возвращаем полностью проверенные матрицы для обучения и прогнозирования.
    return X_train, X_test

## 9. Канонизация feature matrix

В raw-таблице остаются технические datetime-колонки, которые нужны для построения признаков, но не являются признаками модели.

Также `n_events` рассчитывается двумя блоками (`activity` и `temporal`), поэтому после merge pandas может создать `n_events_x` и `n_events_y`. Они несут одну и ту же информацию, поэтому оставляю одну каноническую колонку `n_events`.

В финальную модель попадают только числовые признаки. Это делает вход CatBoost однозначным и полностью совпадает с финальной исследовательской схемой.

In [52]:
# Контролируем сохранение полного набора cookie после feature engineering.
def validate_feature_coverage(
        meta: pd.DataFrame,
        feature_table: pd.DataFrame,
) -> None:
    """Ensure feature engineering neither loses nor invents cookies."""

    # Получаем эталонное множество cookie из исходной таблицы.
    meta_ids = set(meta["cookie_id"])

    # Получаем множество cookie из построенной таблицы признаков.
    feature_ids = set(feature_table["cookie_id"])

    # Ни одна исходная cookie не должна потеряться и не должна появиться искусственно.
    assert meta_ids == feature_ids

    # Дополнительно требуем ровно одну строку признаков на cookie.
    assert feature_table["cookie_id"].is_unique

## 10. Заполнение пропусков и финальные проверки

После агрегации event-level данных cookie без событий не имеют строки в соответствующей groupby-таблице. При `left join` это превращается в `NaN`.

Для event-derived признаков естественное значение отсутствующей активности - `0`, поэтому эти значения заполняются нулями.

Перед обучением дополнительно проверяю:

- отсутствие пропусков;
- отсутствие бесконечных значений;
- одинаковую схему train/test;
- отсутствие `cookie_id` среди признаков.

In [53]:
# Централизуем создание финальной модели, чтобы параметры не дублировались в разных местах notebook.
def create_model() -> CatBoostClassifier:
    """Create the final model selected by walk-forward validation."""

    # Используем выбранный по walk-forward validation CatBoostClassifier для бинарной классификации.
    return CatBoostClassifier(
        # 500 итераций задают число последовательных шагов boosting.
        iterations=500,

        # Глубина 6 ограничивает сложность отдельных деревьев.
        depth=6,

        # Небольшой learning rate делает обновление модели более постепенным.
        learning_rate=0.05,

        # Logloss - функция потерь для бинарной классификации.
        loss_function="Logloss",

        # AUC задается как внутренняя оценочная метрика CatBoost во время обучения.
        eval_metric="AUC",

        # Фиксируем seed, связанный с общей настройкой воспроизводимости проекта.
        random_seed=SEED,

        # Отключаем подробный служебный вывод CatBoost.
        verbose=False,

        # Разрешаем CatBoost использовать доступные ядра процессора.
        thread_count=-1,

        # Запрещаем создание служебных файлов CatBoost рядом с решением.
        allow_writing_files=False,
    )

## 11. Финальная модель

По результатам проведенного исследования финальной моделью выбран `CatBoostClassifier`.

Параметры были зафиксированы до финального обучения:

- `iterations=500`;
- `depth=6`;
- `learning_rate=0.05`;
- `loss_function="Logloss"`;
- `eval_metric="AUC"`;
- `random_seed=42`.

В финальном запуске модель обучается на **всем train**, потому что validation использовалась только для выбора подхода.

In [54]:
# Формирование submission выделено отдельно, чтобы формат результата проверялся до записи на диск.
def build_submission(
        test: pd.DataFrame,
        scores: np.ndarray,
) -> pd.DataFrame:
    """Create and validate the exact two-column competition submission."""

    # Приводим предсказания к числовому массиву фиксированного типа.
    scores = np.asarray(scores, dtype=float)

    # Создаем ровно две требуемые колонки: идентификатор cookie и непрерывный score.
    submission = pd.DataFrame(
        {
            # Сохраняем исходный порядок cookie из test.
            "cookie_id": test["cookie_id"].to_numpy(),
            # Score - вероятность класса 1, то есть оценка принадлежности к автоматизированному трафику.
            "score": scores,
        }
    )

    # Проверяем точное имя и порядок двух колонок.
    assert list(submission.columns) == ["cookie_id", "score"]

    # Количество строк submission должно совпадать с количеством тестовых cookie.
    assert len(submission) == len(test)

    # Одна cookie должна присутствовать в submission ровно один раз.
    assert submission["cookie_id"].is_unique

    # Идентификаторы не могут быть пропущены.
    assert submission["cookie_id"].notna().all()

    # Ни один score не должен быть NaN.
    assert submission["score"].notna().all()

    # Score должен быть конечным числом.
    assert np.isfinite(submission["score"].to_numpy()).all()

    # Проверяем требуемый диапазон вероятности от 0 до 1.
    assert submission["score"].between(0, 1).all()

    # Проверяем, что идентификаторы и их порядок полностью совпадают с test.
    assert submission["cookie_id"].equals(test["cookie_id"])

    # Возвращаем проверенный DataFrame, готовый к сохранению.
    return submission

## 12. Предсказания и submission

Здесь намеренно не применяется порог вроде `0.5`.

Задание просит **не бинарный ответ**, а непрерывный score. Официальная метрика сама определяет рабочий threshold при достижении требуемого recall.

Поэтому в submission записывается непосредственно вероятность класса `1`:

`predict_proba(X_test)[:, 1]`.

In [55]:
# main() связывает все этапы решения в один воспроизводимый pipeline от исходных файлов до submission.
def main() -> None:
    # Загружаем исходные таблицы.
    train, test, events = load_data()

    # До любых вычислений проверяем схему, уникальность ключей и корректность окон.
    validate_input(train, test, events)

    # Оставляем для train только события внутри индивидуальных 24-часовых окон.
    ev_train = events_in_window(events, train)

    # То же самое выполняем для test.
    ev_test = events_in_window(events, test)

    # Повторная проверка подтверждает отсутствие событий за границами разрешенного окна.
    validate_events_in_window(ev_train, ev_test)

    # Выводим размеры исходных таблиц и количество событий внутри окон как итоговый контроль подготовки данных.
    print(f"Train: {train.shape}")
    print(f"Test:  {test.shape}")
    print(f"Events: {events.shape}")
    print(f"Train events in window: {len(ev_train)}")
    print(f"Test events in window:  {len(ev_test)}")

    # Строим сырые признаки уровня cookie для обучения.
    X_train_raw = build_feature_table(train, ev_train)

    # Строим тот же набор признаков для теста.
    X_test_raw = build_feature_table(test, ev_test)

    # Проверяем, что feature engineering не потерял train-cookie.
    validate_feature_coverage(train, X_train_raw)

    # То же проверяем для test-cookie.
    validate_feature_coverage(test, X_test_raw)

    # Приводим обе выборки к единой числовой схеме признаков.
    X_train, X_test, feature_cols = canonicalize_features(
        X_train_raw,
        X_test_raw,
    )

    # Заполняем агрегированные пропуски и проверяем матрицы перед моделью.
    X_train, X_test = fill_and_validate_features(
        X_train,
        X_test,
        feature_cols,
    )

    # Target не должен попасть в признаки: это непосредственно целевая переменная и источник leakage.
    assert "target" not in X_train.columns

    # В test target вообще отсутствует и не должен появиться после объединений.
    assert "target" not in X_test.columns

    # Идентификатор используется только как ключ и не подается модели как числовой сигнал.
    assert "cookie_id" not in feature_cols

    # Извлекаем целевую переменную отдельно от feature matrix.
    y_train = train["target"].astype(int).to_numpy()

    # Создаем финальный CatBoost с зафиксированными параметрами.
    model = create_model()

    # После завершения выбора признаков и модели обучаем ее на всем доступном размеченном train.
    model.fit(X_train[feature_cols], y_train)

    # Получаем вероятность класса 1 для каждой test-cookie; это непрерывный score без выбора порога.
    scores = model.predict_proba(X_test[feature_cols])[:, 1]

    # Проверяем и собираем итоговый формат соревнования.
    submission = build_submission(test, scores)

    # Записываем submission.csv без дополнительного индекса pandas.
    submission.to_csv(SUBMISSION_PATH, index=False)

    # Выводим число использованных признаков и путь к результату для контроля запуска.
    print(f"Features: {len(feature_cols)}")
    print(f"Submission: {SUBMISSION_PATH}")
    print(f"Submission shape: {submission.shape}")
    print(
        "Score range: "
        f"[{submission['score'].min():.6f}, "
        f"{submission['score'].max():.6f}]"
    )
    print("Submission validation: PASS")


# Запускаем main() только при непосредственном выполнении файла, а не при его импорте.
if __name__ == "__main__":
    main()

Train: (11091, 5)
Test:  (4909, 4)
Events: (328905, 14)
Train events in window: 198436
Test events in window:  89690
Features: 37
Submission: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\submission.csv
Submission shape: (4909, 2)
Score range: [0.000487, 0.999696]
Submission validation: PASS


## 13. Итог

Финальный pipeline можно воспроизвести одной последовательностью:

`raw events → фильтрация по 24h window → cookie-level aggregation → numeric feature matrix → CatBoost → continuous score → submission.csv`

В результате используется **37 числовых признаков**, построенных из разрешенного окна наблюдения.

### Что особенно важно проверить перед отправкой

1. `data/train.csv`, `data/test.csv` и `data/events.csv.gz` находятся в ожидаемой папке.
2. Notebook выполнен сверху вниз без ручного изменения данных.
3. `submission.csv` содержит ровно `cookie_id, score`.
4. Число строк совпадает с `test.csv`.
5. Нет дубликатов `cookie_id`, пропусков или значений `score` вне `[0, 1]`.
6. В submission не применяется вручную выбранный classification threshold.

> Примечание: внутренний результат `P@R >= 0.70 = 0.4575` относится к walk-forward validation на train и не является гарантией результата на скрытом тесте.